# L6.1 · Flow matching: el problema

### El mapa de la serie

| Notebook | Qué abre |
|---|---|
| L6.1 | El problema: por qué la regresión cae en el medio, y por qué tampoco sirve aprender "ruido → acción" de un salto |
| L6.2 | El camino recto y la brújula: qué velocidad aprende la red y cómo se entrena |
| L6.3 | La inferencia: caminar desde el ruido de a pasitos |
| L6.4 | Condicionar y generar bloques: una política de verdad |
| L6.5 | El código real de LeRobot (SmolVLA, π0), lado a lado con el nuestro |

Todo es 2D y chico: cada red entrena en segundos en la CPU y se puede dibujar
todo lo que pasa adentro. El juguete (los datos y el dibujo de la escena) vive
en `toolkit/flow/toy.py`; lo que se quiere entender (las pérdidas, el
entrenamiento, los pasos) está escrito acá, a mano.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from toolkit.flow import toy

plt.rcParams["figure.dpi"] = 110
# Redes chiquitas: con los 16 núcleos PyTorch pierde más tiempo coordinándolos
# que calculando (26 s contra 2 s para el mismo entrenamiento).
torch.set_num_threads(4)
torch.manual_seed(0)

## 1. El juguete: esquivar un árbol

El robot está parado y tiene un árbol adelante. Juntamos 400 demostraciones:
cada una es **el punto por donde la persona pasó al costado del árbol**. La
mitad fue por la izquierda y la mitad por la derecha. Nadie fue por el medio,
porque en el medio está el tronco.

Es la versión más chica del problema de siempre: hay dos formas válidas de
hacer la tarea, y ninguna forma válida "entre" las dos.

In [ ]:
demos = toy.sample_demos(400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
toy.draw_scene(ax1, demos, title="400 demostraciones")
ax2.hist(demos[:, 0], bins=60, color="gray")
ax2.axvspan(-toy.TREE_RADIUS, toy.TREE_RADIUS, color="forestgreen", alpha=0.2, label="el árbol")
ax2.set_xlabel("posición lateral (izquierda < 0 < derecha)")
ax2.set_ylabel("demostraciones")
ax2.set_title("dos picos y un valle vacío")
ax2.legend()
plt.tight_layout()

## 2. Lo que hace una regresión

La forma directa de aprender una política: una red que mira la observación y
devuelve **un** punto, entrenada para que ese punto quede lo más cerca posible
de las demostraciones. "Lo más cerca posible" medido con error cuadrático
(MSE) es la elección de siempre.

Acá la observación es siempre la misma (el mismo árbol en el mismo lugar), así
que la red tiene que elegir **un solo punto** que quede cerca de todas las
demostraciones a la vez. La entrenamos y miramos por dónde se mueve su
predicción mientras aprende.

In [ ]:
observation = torch.tensor([toy.TREE_CENTER])  # lo único que ve la política: dónde está el árbol


def make_policy():
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))


def train_regression(demos, loss_fn, steps=600, seed=0):
    """Entrena la política y devuelve su predicción después de cada paso."""
    torch.manual_seed(seed)
    policy = make_policy()
    optimizer = torch.optim.Adam(policy.parameters(), lr=1e-2)
    path = []
    for step in range(steps):
        batch = demos[torch.randint(len(demos), (128,))]
        prediction = policy(observation).expand_as(batch)
        loss = loss_fn(prediction, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        path.append(policy(observation).detach()[0].numpy())
    return np.array(path)


def mse(prediction, target):
    return ((prediction - target) ** 2).mean()


path = train_regression(demos, mse)

fig, ax = plt.subplots(figsize=(6, 4.5))
toy.draw_scene(ax, demos, title="dónde termina la regresión con MSE", alpha=0.25)
ax.plot(path[:, 0], path[:, 1], color="red", lw=1, alpha=0.6, label="la predicción mientras entrena")
ax.scatter(*path[-1], marker="X", s=150, color="red", zorder=5, label="al final")
ax.legend(loc="lower left", fontsize=8)
print("predicción final  :", path[-1].round(3))
print("promedio de demos :", demos.mean(0).numpy().round(3))

Termina **adentro del tronco**: el robot chocaría. Y no es que a la red le
falte capacidad. El punto que minimiza el error cuadrático contra un montón de
demostraciones es, siempre, **su promedio**: lo dice la cuenta, no es una
limitación de esta red. Una red más grande termina en el mismo lugar. El
promedio de "por la izquierda" y "por la derecha" es "de frente contra el
árbol".

### ¿Y con L1, como ACT?

ACT no usa error cuadrático sino **error absoluto** (L1), y el paper dice que
le fue mejor. Con L1 el óptimo ya no es el promedio sino la **mediana** de
cada coordenada. Probamos con mitad y mitad, y con 70 % por la izquierda,
cinco entrenamientos de cada uno:

In [ ]:
def l1(prediction, target):
    return (prediction - target).abs().mean()


fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, fraction in zip(axes, [0.5, 0.7]):
    some_demos = toy.sample_demos(400, left_fraction=fraction, seed=1)
    toy.draw_scene(ax, some_demos, title=f"L1 · {fraction:.0%} por la izquierda", alpha=0.25)
    for seed in range(5):
        ax.scatter(*train_regression(some_demos, l1, seed=seed)[-1], marker="X", s=110, color="red", zorder=5)
plt.tight_layout()

Con mitad y mitad, L1 termina **en cualquier lugar del valle**, varias veces
adentro del árbol: con un empate, cualquier punto entre los dos grupos es una
mediana igual de buena, y dónde termina depende de la suerte del
entrenamiento. Con 70 % por la izquierda, en cambio, elige la izquierda...
**siempre**, los cinco entrenamientos. No promedia, pero tampoco sortea: nunca
va a ir por la derecha, aunque el 30 % de la gente lo hizo. Si en cierta
situación la estrategia minoritaria es la única que sirve, la política no la
tiene.

Ninguna de las dos hace lo que queremos.

## 3. Lo que queremos: sortear

Lo que queremos es una máquina que **genere muestras con las frecuencias
correctas**: que el 70 % de las veces vaya por la izquierda, el 30 % por la
derecha, y nunca por el medio.

Sortear de una distribución rara (dos picos con un valle) es difícil. Sortear
de una distribución fácil, como una campana, es trivial: `torch.randn`.
Entonces la idea es **sortear ruido fácil y pasarlo por una función** que lo
convierta en una demostración posible. Ruido distinto, demostración distinta.

¿Existe esa función? En 1D sí, y se arma a mano: se **ordena** el ruido, se
**ordenan** las demostraciones, y se empareja por orden. El ruido más chico va
con la demostración más a la izquierda, el segundo con la segunda, y así.
Probamos con la coordenada lateral:

In [ ]:
lateral = demos[:, 0].numpy()
noise = np.random.default_rng(0).normal(size=len(lateral))
sorted_noise, sorted_lateral = np.sort(noise), np.sort(lateral)

fresh_noise = np.random.default_rng(1).normal(size=4000)
generated = np.interp(fresh_noise, sorted_noise, sorted_lateral)  # la función: seguir el emparejamiento

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].hist(fresh_noise, bins=60, color="gray")
axes[0].set_title("1 · ruido: una campana")
axes[0].set_xlabel("ruido")
axes[1].plot(sorted_noise, sorted_lateral, color="black")
axes[1].axhspan(-toy.TREE_RADIUS, toy.TREE_RADIUS, color="forestgreen", alpha=0.2)
axes[1].set_xlabel("ruido")
axes[1].set_ylabel("posición lateral")
axes[1].set_title("2 · la función (emparejar por orden)")
axes[2].hist(generated, bins=60, color="gray")
axes[2].axvspan(-toy.TREE_RADIUS, toy.TREE_RADIUS, color="forestgreen", alpha=0.2)
axes[2].set_title("3 · ruido nuevo pasado por la función")
axes[2].set_xlabel("posición lateral")
plt.tight_layout()
print(f"muestras generadas que caen en el árbol: {np.mean(np.abs(generated) < toy.TREE_RADIUS):.2%}")

Funciona. La función es una escalera con un **salto casi vertical** justo en
el medio. Ese salto es donde "decide": el ruido de un lado del salto va a la
izquierda y el del otro lado a la derecha, y como el salto es casi vertical,
casi nada cae en el valle. Además respeta las proporciones: si hubiera 70 % de
demostraciones a la izquierda, el salto estaría corrido y el 70 % del ruido
caería de ese lado. La función no tiene nada de azar: el azar está todo en el
ruido.

Esa es la máquina que queremos. La pregunta es cómo **aprenderla** con una red
cuando no estamos en 1D.

## 4. ¿Por qué no aprenderla directo?

Lo obvio: una red `f(ruido) → demostración`, entrenada con MSE sobre pares
(ruido, demostración). ¿Qué pares? En cada paso sorteamos ruido nuevo y lo
emparejamos con demostraciones cualquiera, al azar:

In [ ]:
def train_map(make_pairs, dim, steps=2000, seed=0):
    """Entrena f(ruido) → demostración con MSE sobre los pares que arma `make_pairs`."""
    torch.manual_seed(seed)
    f = nn.Sequential(nn.Linear(dim, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, dim))
    optimizer = torch.optim.Adam(f.parameters(), lr=3e-3)
    for step in range(steps):
        noise, targets = make_pairs(256)
        loss = ((f(noise) - targets) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return f


def random_pairs(n):
    """Ruido nuevo, emparejado con demostraciones cualquiera."""
    return torch.randn(n, 2), demos[torch.randint(len(demos), (n,))]


f_random = train_map(random_pairs, dim=2)
with torch.no_grad():
    samples = f_random(torch.randn(1000, 2))

fig, ax = plt.subplots(figsize=(6, 4.5))
toy.draw_scene(ax, demos, title="f(ruido) entrenada con parejas al azar", alpha=0.2)
ax.scatter(samples[:, 0], samples[:, 1], s=6, color="red", label="1000 muestras generadas")
ax.legend(loc="lower left", fontsize=8)

Las mil muestras caen **otra vez en el tronco**. Es el problema de la sección
2, disfrazado: si el ruido no dice nada sobre a qué demostración corresponde,
lo mejor que puede hacer la red ante cualquier ruido es devolver el promedio.
El ruido entra, pero la red aprende a ignorarlo.

En 1D lo resolvimos emparejando por orden. Si le damos a la red esos pares
ordenados, en cambio:

In [ ]:
pairs_noise = torch.tensor(sorted_noise, dtype=torch.float32)[:, None]
pairs_lateral = torch.tensor(sorted_lateral, dtype=torch.float32)[:, None]


def sorted_pairs(n):
    """Los pares de la sección 3: cada ruido con la demostración que le toca por orden."""
    idx = torch.randint(len(pairs_noise), (n,))
    return pairs_noise[idx], pairs_lateral[idx]


f_sorted = train_map(sorted_pairs, dim=1)
grid = torch.linspace(-3, 3, 300)[:, None]
with torch.no_grad():
    learned = f_sorted(grid)[:, 0]
    out_sorted = f_sorted(torch.randn(4000, 1))[:, 0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(sorted_noise, sorted_lateral, color="black", lw=1, ls="--", label="la función de la sección 3")
axes[0].plot(grid[:, 0], learned, color="red", label="lo que aprendió la red")
axes[0].set_xlabel("ruido")
axes[0].set_ylabel("posición lateral")
axes[0].legend(fontsize=8)
axes[1].hist(out_sorted, bins=60, color="red", alpha=0.7)
axes[1].axvspan(-toy.TREE_RADIUS, toy.TREE_RADIUS, color="forestgreen", alpha=0.2)
axes[1].set_title("muestras de la red entrenada con pares ordenados")
axes[1].set_xlabel("posición lateral")
plt.tight_layout()
print(f"muestras que caen en el árbol: {np.mean(np.abs(out_sorted) < toy.TREE_RADIUS):.2%}")

Ahora sí: la red copió la escalera, salto incluido, y las muestras salen con
dos picos. Cerca del 1 % cae en el valle: una red es una función suave y no
puede hacer un salto perfectamente vertical, así que el ruido que cae justo en
el salto termina en el medio. Es un detalle que va a volver.

Entonces el problema no es la red: es **saber qué ruido va con qué
demostración**. En 1D se ordena y listo. Pero ¿cómo se "ordena" en 2D? ¿Y en
un bloque de ACT, que son 48 acciones × 26 articulaciones = 1248 números? No
hay un orden natural.

## Lo que nos llevamos

- La regresión con MSE cae en el promedio, que acá es el tronco. Con L1 elige
  la mayoría, siempre. Ninguna de las dos **sortea**.
- Lo que queremos es una función que convierta ruido fácil de sortear en
  demostraciones posibles, con las proporciones correctas. Existe: en 1D se
  arma ordenando.
- Aprenderla directo con parejas al azar vuelve a caer en el promedio. El
  problema es que no sabemos qué ruido va con qué demostración.

**La idea de flow matching (L6.2):** en vez de aprender el salto entero de una
vez, aprender **pasitos**. Con parejas al azar, adivinar el destino es
imposible y la red promedia destinos: el tronco. Pero si en cada punto del
camino sólo tiene que adivinar **para dónde moverse un poquito**, promediar
direcciones resulta inofensivo. Eso es lo que vamos a ver.

## Referencias

- Chi et al., [Diffusion Policy](https://arxiv.org/abs/2303.04137) (2023): el
  mismo problema en una tarea real (empujar una T por la izquierda o por la
  derecha) y por qué la regresión lo rompe.
- Zhao et al., [ACT](https://arxiv.org/abs/2304.13705) (2023): de donde sale
  el L1.
- Lipman et al., [Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747)
  (2023): el paper que le da el nombre.
- Liu et al., [Flow Straight and Fast](https://arxiv.org/abs/2209.03003)
  (2023): la misma idea con caminos rectos (*rectified flow*).
- Lipman et al., [Flow Matching Guide and Code](https://arxiv.org/abs/2412.06264)
  (2024): el tutorial largo de los autores, con código.